# 15.6 Hashing

**Prerequisites:** 15.1 Complexity Analysis, 15.2 Python's Built-ins, 05 OOPs  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- How a hash table turns a key into a memory location
- **Collisions**, and the two ways to resolve them
- Building a working hash map **from scratch**
- **Load factor** and resizing - where the amortised O(1) comes from
- What makes a good hash function, and why Python randomises string hashes
- 🔴 When O(1) degrades to O(n)
- The four hashing patterns that solve most interview problems
- Designing composite keys with `tuple` and `frozenset`
- Interview questions, worked

---

## The problem hashing solves

You have a key — `"user:42"` — and you want its value **immediately**, without searching.

| Approach | Lookup |
|---|---|
| Scan a list of pairs | O(n) |
| Binary search a sorted list | O(log n), and insertion is O(n) (**15.2**) |
| **Hash table** | **O(1) average** |

### The idea

Do not search. **Compute** where it lives.

```
   "user:42"  ──hash()──>  8374625193  ──% 8──>  slot 1
                                                    │
        buckets:  [0] [1] [2] [3] [4] [5] [6] [7]    │
                       ▲                             │
                       └─────────────────────────────┘
                       store (key, value) here
```

Lookup repeats the same arithmetic and lands in the same bucket. The table's **size is irrelevant** to that work — hence O(1).

> **The catch, and the entire rest of this notebook:** two different keys can produce the same slot. That is a **collision**, and how you handle it determines whether your table stays O(1).

In [ ]:
# Python exposes the first step directly.
for key in ("user:42", "user:43", 42, 3.5, (1, 2), True):
    print(f"  hash({key!r:<10}) = {hash(key)}")

print("\nand the second step is just modulo:")
TABLE_SIZE = 8
for key in ("user:42", "user:43", "user:44", "session"):
    print(f"  {key:<10} -> slot {hash(key) % TABLE_SIZE}")

print("\n🔴 Two things worth noticing:")
print("   hash(True) == hash(1) - because True == 1. Equal objects MUST")
print("   hash equally, so {1: 'a', True: 'b'} has ONE entry:")
print("      ", {1: "a", True: "b"})
print("\n   And string hashes change between runs - see the section below.")

## Collisions, and the two strategies

With 8 buckets and 9 keys, a collision is *guaranteed*. Long before that it is merely *likely* — the birthday paradox means collisions appear far sooner than intuition suggests.

### Separate chaining

Each bucket holds a **list** of entries.

```
   [0] -> None
   [1] -> [("user:42", A), ("session", B)]     both hashed to 1
   [2] -> [("user:43", C)]
```

Simple, degrades gracefully, and tolerates a high load factor. Costs an extra list per bucket.

### Open addressing

One entry per bucket; on collision, **probe** for another slot.

```
   slot 1 taken -> try 2 -> try 3 -> ...      (linear probing)
```

Better memory locality — everything is in one array — but it degrades badly as the table fills, and deletion needs a tombstone marker rather than simply emptying a slot.

| | Chaining | Open addressing |
|---|---|---|
| Memory | extra list per bucket | one flat array |
| Cache behaviour | poorer | ✅ better |
| Tolerates high load | ✅ yes | no — degrades sharply |
| Deletion | trivial | needs tombstones |
| **CPython uses** | | ✅ **open addressing** |

We will build the chaining version, because it is clearer, then discuss what CPython actually does.

In [ ]:
class HashMap:
    """A hash map with separate chaining. Educational, not production.

    Average O(1) get/set/delete; O(n) worst case if everything collides.
    """

    def __init__(self, capacity=8):
        self._capacity = capacity
        self._buckets = [[] for _ in range(capacity)]
        self._size = 0
        self.collisions = 0            # instrumentation
        self.resizes = 0

    def _slot(self, key):
        return hash(key) % self._capacity

    def put(self, key, value):
        bucket = self._buckets[self._slot(key)]
        if bucket:
            self.collisions += 1
        for i, (existing, _) in enumerate(bucket):
            if existing == key:        # 🔴 == , not `is` - equal keys replace
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self._size += 1
        if self._size / self._capacity > 0.75:      # the load factor
            self._grow()

    def get(self, key, default=None):
        for existing, value in self._buckets[self._slot(key)]:
            if existing == key:
                return value
        return default

    def delete(self, key):
        bucket = self._buckets[self._slot(key)]
        for i, (existing, _) in enumerate(bucket):
            if existing == key:
                del bucket[i]
                self._size -= 1
                return True
        return False

    def _grow(self):
        """Double the capacity and REHASH everything - slots depend on capacity."""
        self.resizes += 1
        old = self._buckets
        self._capacity *= 2
        self._buckets = [[] for _ in range(self._capacity)]
        self._size = 0
        for bucket in old:
            for key, value in bucket:
                self.put(key, value)

    def __len__(self):
        return self._size

    def __contains__(self, key):
        sentinel = object()
        return self.get(key, sentinel) is not sentinel

    def bucket_sizes(self):
        return [len(b) for b in self._buckets]


table = HashMap(capacity=8)
for i in range(6):
    table.put(f"user:{i}", f"payload-{i}")

print("stored      :", len(table))
print("get user:3  :", table.get("user:3"))
print("get missing :", table.get("nope", "<default>"))
print("'user:1' in :", "user:1" in table)
print("bucket sizes:", table.bucket_sizes())
print("resizes     :", table.resizes)

table.put("user:3", "replaced")
print("\nafter replacing user:3 ->", table.get("user:3"), "| size still", len(table))
print("delete user:0:", table.delete("user:0"), "| size now", len(table))
print("delete absent:", table.delete("user:0"))

## Load factor - where amortised O(1) comes from

```
                     number of entries
    load factor  =  ───────────────────
                     number of buckets
```

As it rises, buckets hold more entries and lookups take longer. So the table **resizes** when it crosses a threshold — typically 0.66 to 0.75 — doubling capacity and rehashing everything.

🔴 **Rehashing is mandatory, not an optimisation.** A slot is `hash(key) % capacity`; change the capacity and every key belongs somewhere new. Skipping it silently loses entries.

Resizing is O(n), but it happens geometrically less often as the table grows — exactly the `list.append` argument from **15.1**. That is why insertion is **amortised** O(1) rather than plain O(1).

In [ ]:
table = HashMap(capacity=8)
print(f"{'inserted':>9}{'capacity':>10}{'load':>8}{'resizes':>9}{'max bucket':>12}")
print("-" * 48)
for i in range(1, 65):
    table.put(f"key-{i}", i)
    if i in (1, 6, 7, 12, 13, 24, 25, 48, 49, 64):
        load = len(table) / table._capacity
        print(f"{i:>9}{table._capacity:>10}{load:>8.2f}{table.resizes:>9}"
              f"{max(table.bucket_sizes()):>12}")

print("\n  Capacity doubles whenever the load factor passes 0.75, and the")
print("  longest bucket stays short - which is what keeps lookups O(1).")

print("\n  every key still resolves after all that rehashing:")
print("   ", all(table.get(f"key-{i}") == i for i in range(1, 65)))

## 🔴 When O(1) becomes O(n)

The average case assumes keys spread evenly. Make them collide and a hash table becomes a linked list with extra steps.

The next cell builds a deliberately terrible hash function — one returning a constant — so every key lands in the same bucket.

This is not only theoretical. Before Python 3.3, an attacker could craft thousands of strings that all hashed to the same bucket and send them as HTTP form fields, turning an O(1) dictionary into an O(n) one and stalling the server: a **hash-collision denial-of-service**.

**The fix, and why your string hashes change between runs:** Python now seeds string hashing with a random value per process. An attacker cannot predict the buckets. You can see this yourself — `hash("a")` differs across interpreter runs, and setting `PYTHONHASHSEED=0` disables the randomisation.

🔴 That is also why **`hash()` values must never be persisted** to a file or database. They will not match on the next run.

In [ ]:
class TerribleKey:
    """A legal but useless hash: every instance collides with every other."""

    def __init__(self, name):
        self.name = name

    def __hash__(self):
        return 1                      # legal, and catastrophic

    def __eq__(self, other):
        return isinstance(other, TerribleKey) and self.name == other.name


import time

N = 2_000

good = {f"key-{i}": i for i in range(N)}
started = time.perf_counter()
for i in range(N):
    _ = good[f"key-{i}"]
good_time = time.perf_counter() - started

bad = {TerribleKey(f"key-{i}"): i for i in range(N)}
started = time.perf_counter()
for i in range(N):
    _ = bad[TerribleKey(f"key-{i}")]
bad_time = time.perf_counter() - started

print(f"{N:,} lookups\n")
print(f"  well-distributed hash : {good_time * 1000:9.2f} ms   O(1) each")
print(f"  everything collides   : {bad_time * 1000:9.2f} ms   O(n) each")
print(f"  ratio                 : {bad_time / good_time:9,.0f}x")
print("\n  Same dict, same size. Only the hash function changed.")
print("  This is the worst case in the complexity table, made real.")

import sys
print(f"\nhash randomisation this process: PYTHONHASHSEED not fixed ->")
print(f"  hash('a') = {hash('a')}")
print("  Run this notebook again and that number will differ.")
print("  🔴 Never store a hash() value anywhere it must survive a restart.")

## Writing `__hash__` and `__eq__` correctly

The contract has one rule that matters:

> **If `a == b`, then `hash(a) == hash(b)`.**

The reverse need not hold — unequal objects may share a hash; that is just a collision.

| If you... | Then... |
|---|---|
| define `__eq__` only | Python sets `__hash__ = None` → **unhashable** |
| define both consistently | ✅ usable as a dict key |
| define them inconsistently | 🔴 entries silently go missing |
| hash on a **mutable** field | 🔴 mutating it orphans the entry (**15.2**) |

**The safe recipe:** hash a tuple of the same immutable fields you compare on.

```
    def __hash__(self):
        return hash((self.host, self.port))
    def __eq__(self, other):
        return (self.host, self.port) == (other.host, other.port)
```

Easier still: a **frozen dataclass** (**5.3**) generates both correctly for you.

In [ ]:
from dataclasses import dataclass


# ---- 🔴 __eq__ without __hash__ ----
class OnlyEq:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return isinstance(other, OnlyEq) and self.value == other.value


try:
    {OnlyEq(1): "x"}
except TypeError as exc:
    print("defining __eq__ alone:", exc)
    print("  ^ Python refuses rather than let you break the contract.\n")


# ---- 🔴 inconsistent: equal objects with different hashes ----
class Inconsistent:
    counter = 0

    def __init__(self, value):
        self.value = value
        Inconsistent.counter += 1
        self.serial = Inconsistent.counter   # differs per instance

    def __eq__(self, other):
        return isinstance(other, Inconsistent) and self.value == other.value

    def __hash__(self):
        return hash(self.serial)             # 🔴 not based on `value`


a, b = Inconsistent("same"), Inconsistent("same")
registry = {a: "stored"}
print("a == b            :", a == b)
print("hash(a) == hash(b):", hash(a) == hash(b))
print("b in registry     :", b in registry, " <- 🔴 equal, but not found")
print("  The lookup checked the wrong bucket and never compared them.\n")


# ---- ✅ the safe recipe ----
@dataclass(frozen=True)
class Endpoint:
    host: str
    port: int


x, y = Endpoint("db-1", 5432), Endpoint("db-1", 5432)
print("frozen dataclass:")
print("  equal          :", x == y)
print("  same hash      :", hash(x) == hash(y))
print("  found by a twin:", y in {x: "connection-pool"})
print("  immutable      :", end=" ")
try:
    x.port = 1
except Exception as exc:
    print(type(exc).__name__, "- so the hash can never change")

---

# The four hashing patterns

Recognising these solves a large share of interview problems.

| Pattern | Shape | Turns |
|---|---|---|
| **1. Frequency count** | `Counter(data)` | O(n²) counting → O(n) |
| **2. Seen set** | `if x in seen` | O(n²) duplicate search → O(n) |
| **3. Complement lookup** | `if target - x in seen` | O(n²) pair search → O(n) |
| **4. Grouping by a computed key** | `groups[key(x)].append(x)` | O(n²·k) → O(n·k) |

All four are the same trade: **O(n) memory buys you O(n) time instead of O(n²)** (**15.1**).

In [ ]:
from collections import Counter, defaultdict

# ---- 1. frequency count ----
log = "deploy build deploy test build deploy release".split()
print("1. frequency  :", Counter(log).most_common(2))


def first_non_repeating(text):
    """Two passes, O(n) - not the O(n^2) nested scan."""
    counts = Counter(text)
    for char in text:
        if counts[char] == 1:
            return char
    return None


print("   first unique in 'swiss' ->", first_non_repeating("swiss"))

# ---- 2. seen set ----


def first_duplicate(data):
    seen = set()
    for value in data:
        if value in seen:          # O(1) - 15.2
            return value
        seen.add(value)
    return None


print("\n2. seen set   : first duplicate in [3,1,4,1,5] ->",
      first_duplicate([3, 1, 4, 1, 5]))

# ---- 3. complement lookup ----


def two_sum(data, target):
    seen = {}
    for index, value in enumerate(data):
        if target - value in seen:
            return (seen[target - value], index)
        seen[value] = index
    return None


print("\n3. complement : two_sum([2,7,11,15], 9) ->", two_sum([2, 7, 11, 15], 9))

# ---- 4. grouping by a computed key ----


def group_anagrams(words):
    """The key is what makes anagrams identical: their sorted letters."""
    groups = defaultdict(list)
    for word in words:
        groups[tuple(sorted(word))].append(word)      # tuple: hashable
    return list(groups.values())


print("\n4. grouping   :", group_anagrams(
    ["eat", "tea", "tan", "ate", "nat", "bat"]))
print("   The insight is choosing a key that collides EXACTLY when you")
print("   want it to. Sorted letters do that for anagrams.")
print("   A Counter of letters works too and is O(k) instead of O(k log k).")

## Composite and canonical keys

A key does not have to be a string. Anything hashable works, and choosing the right key is often the whole solution.

| Need | Key |
|---|---|
| Several fields together | `tuple` — `(host, port)` |
| Order-independent group | `frozenset` — `{a, b}` == `{b, a}` |
| Anagram group | `tuple(sorted(word))` |
| Case/space-insensitive | normalise first: `text.strip().casefold()` |
| A whole row | `tuple(row)` — lists are unhashable |

🔴 **`str.casefold()`, not `str.lower()`**, for case-insensitive keys. `lower()` gets some non-English text wrong — German `"ß"` lowercases to itself but casefolds to `"ss"`, so `"STRASSE"` and `"Straße"` only match under `casefold`.

In [ ]:
# ---- tuple keys ----
connections = {("db-1", 5432): "pool-a", ("db-2", 5432): "pool-b"}
print("tuple key      :", connections[("db-1", 5432)])

# ---- frozenset: order-independent ----
friendships = {frozenset({"alice", "bob"}): "since 2019"}
print("frozenset key  :", friendships[frozenset({"bob", "alice"})])
print("  ^ the same pair in either order finds the same entry")

# ---- lists are unhashable; tuples are not ----
try:
    {[1, 2]: "x"}
except TypeError as exc:
    print("\nlist as a key  :", exc)
print("tuple as a key : fine ->", {(1, 2): "x"})

# ---- normalising ----
print("\ncase-insensitive keys:")
print("  'STRASSE'.lower()    ==", repr("STRASSE".lower()))
print("  'Straße'.lower()     ==", repr("Straße".lower()))
print("  lower() matches      :", "STRASSE".lower() == "Straße".lower())
print("  casefold() matches   :", "STRASSE".casefold() == "Straße".casefold())
print("\n  🔴 Use casefold() for case-insensitive keys, not lower().")

## What CPython actually does

Worth knowing, because interviewers sometimes push here.

| | CPython `dict` |
|---|---|
| Collision strategy | **open addressing**, with a perturbation-based probe sequence |
| Load factor | resizes at about **2/3** full |
| Layout | **compact** since 3.6 — a dense entries array plus a sparse index array |
| Ordering | insertion order, a *consequence* of the compact layout, guaranteed from 3.7 |
| String hashing | randomised per process (`PYTHONHASHSEED`) |

The compact layout is why dicts got both smaller and ordered in the same release: entries are appended to a dense array in insertion order, and the sparse array holds only indices into it. Iteration walks the dense array, so it comes out in insertion order — for free.

> **`set` is not a `dict` without values.** It uses a different probing strategy tuned for membership testing, and it does **not** guarantee ordering.

In [ ]:
import sys

print("a dict is smaller than a set of the same keys, thanks to compaction:")
keys = [f"key-{i}" for i in range(1_000)]
as_dict = {k: None for k in keys}
as_set = set(keys)
print(f"  dict: {sys.getsizeof(as_dict):>8,} bytes")
print(f"  set : {sys.getsizeof(as_set):>8,} bytes")

print("\ndict preserves insertion order; set does not promise anything:")
ordered = {}
for word in ("zebra", "apple", "mango"):
    ordered[word] = None
print("  dict:", list(ordered))
print("  set :", list({"zebra", "apple", "mango"}), " <- do not rely on this")

print("\nresizing is visible as a jump in size:")
growing = {}
previous = sys.getsizeof(growing)
jumps = []
for i in range(200):
    growing[i] = i
    size = sys.getsizeof(growing)
    if size != previous:
        jumps.append(i + 1)
        previous = size
print("  resized after inserting:", jumps)
print("  ^ the same geometric pattern as list growth in 15.1 -")
print("    which is where the amortised O(1) comes from.")

## Interview questions

**1. How does a hash table achieve O(1) lookup?**
> Hash the key to compute a bucket index, then probe or chain on collision. Constant work when collisions are rare. Mention that the worst case is O(n).

**2. What happens on a collision?**
> Chaining stores a list per bucket; open addressing probes for another slot. CPython uses open addressing.

**3. What is a load factor and why resize?**
> Entries ÷ buckets. Past ~2/3 collisions rise and lookups degrade, so the table doubles and rehashes. Resizing is O(n) but geometrically rare, giving amortised O(1).

**4. Why must you rehash on resize?**
> The slot is `hash(key) % capacity`. Change the capacity and every key belongs elsewhere.

**5. What makes a good hash function?**
> Deterministic, fast, and spreading keys uniformly. Equal objects must hash equally.

**6. Why do Python string hashes change between runs?**
> Randomised seeding, to prevent hash-collision denial-of-service. So never persist a `hash()` value.

**7. Two Sum.** *(implemented above)*
> Complement lookup in one pass: O(n) time, O(n) space, versus O(n²) brute force.

**8. Group anagrams.** *(implemented above)*
> Group by sorted letters, or by a letter-count tuple. The insight is choosing a key that collides exactly when you want.

**9. First non-repeating character.** *(implemented above)*
> `Counter`, then one pass in original order. O(n) instead of O(n²).

**10. Longest consecutive sequence in an unsorted array, in O(n).**
> Put everything in a set, then only start counting from values whose predecessor is absent — so each run is walked once. Sorting would be O(n log n).

**11. Why can't a list be a dict key?**
> It is mutable, so its hash could change and the entry would become unreachable. Use a tuple.

**12. Design a data structure with O(1) insert, delete and get-random.**
> A list for the values plus a dict from value to index. To delete, swap the target with the last element and pop — O(1). Random is `random.choice` on the list.

In [ ]:
# Questions 10 and 12, because both are non-obvious.
import random


def longest_consecutive(data):
    """O(n): only start counting where a run BEGINS."""
    values = set(data)
    best = 0
    for value in values:
        if value - 1 in values:
            continue                 # not the start of a run - skip
        length = 1
        while value + length in values:
            length += 1
        best = max(best, length)
    return best


for sample in ([100, 4, 200, 1, 3, 2], [0, 0, 1], [], [5]):
    print(f"  longest consecutive in {str(sample):<24} = {longest_consecutive(sample)}")
print("\n  The `continue` is what keeps it O(n): each run is walked exactly")
print("  once, from its smallest member. Without it, it would be O(n^2).")


class RandomSet:
    """O(1) insert, delete and get_random."""

    def __init__(self):
        self._values = []
        self._index = {}

    def insert(self, value):
        if value in self._index:
            return False
        self._index[value] = len(self._values)
        self._values.append(value)
        return True

    def delete(self, value):
        if value not in self._index:
            return False
        # 🔴 The trick: swap with the LAST element, then pop. O(1) instead
        # of the O(n) that removing from the middle would cost (15.2).
        position = self._index[value]
        last = self._values[-1]
        self._values[position] = last
        self._index[last] = position
        self._values.pop()
        del self._index[value]
        return True

    def get_random(self, rng):
        return rng.choice(self._values)

    def __len__(self):
        return len(self._values)


bag = RandomSet()
for value in ("a", "b", "c", "d"):
    bag.insert(value)
print("\n  inserted 4, duplicate insert:", bag.insert("a"))
print("  delete 'b'                  :", bag.delete("b"), "| size", len(bag))
print("  delete absent               :", bag.delete("zzz"))
rng = random.Random(15)
print("  three random picks          :", [bag.get_random(rng) for _ in range(3)])
print("\n  A list alone cannot delete in O(1); a dict alone cannot pick a")
print("  uniform random element. Together they do both.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Defining `__eq__` without `__hash__`.** The class becomes unhashable - Python does this deliberately rather than let you break the contract.
2. 🔴 **Hashing on a mutable field.** Mutate it and the entry is unreachable and un-deletable (**15.2**).
3. 🔴 **Inconsistent `__eq__` and `__hash__`.** Equal objects that hash differently are silently not found.
4. 🔴 **Persisting a `hash()` value.** String hashing is randomised per process; it will not match after a restart.
5. **Forgetting to rehash on resize.** The slot depends on the capacity.
6. **Assuming a set preserves order.** `dict` does since 3.7; `set` never promised it.
7. **Using a list as a key.** Unhashable - use a tuple.
8. **Using `lower()` for case-insensitive keys.** `casefold()` is the correct one.
9. **Assuming O(1) is guaranteed.** It is the average; adversarial or terrible hashes give O(n).

## Best Practices

- Reach for a `set` or `dict` the moment you find yourself searching inside a loop.
- Use `frozen=True` dataclasses for keys - correct `__eq__` and `__hash__` for free.
- Hash a tuple of exactly the fields you compare on.
- Use `Counter` for frequencies and `defaultdict` for grouping.
- Choose keys that collide exactly when you want them to - sorted letters, normalised text, tuples of fields.
- Use `frozenset` when the key is an unordered group.
- Normalise text keys once, at the boundary, not at every lookup.
- State both time *and* space when you propose a hashing solution - it is always a space-for-time trade.

## Practice Exercises

Try these before moving on.

1. Add open addressing with linear probing to `HashMap` as an alternative to chaining. How do you delete without breaking the probe sequence?
2. 🔴 Instrument `HashMap` to report the average bucket length as the load factor rises. At what load does it stop looking like O(1)?
3. Change the resize threshold from 0.75 to 0.95 and re-run the growth cell. What happens to the maximum bucket size?
4. Implement `group_anagrams` with a 26-length count tuple instead of sorting. Compare the complexity for long words.
5. Write a `CaseInsensitiveDict` that normalises keys with `casefold()` on both get and set, while preserving the original casing for display.
6. Implement 'find all pairs summing to k' with a hash map, being careful about duplicates and about not pairing an element with itself.
7. 🔴 Build a class whose `__hash__` uses a mutable field, put it in a dict, mutate it, and then try to delete it. Explain precisely why it cannot be removed.
8. Solve 'longest consecutive sequence' by sorting instead. Compare the complexity, and say when you would prefer the sort.